# Resume Ranking System — End-to-End Demo

This notebook demonstrates the full pipeline:
1. Setup and configuration
2. Data loading and preprocessing
3. Skill extraction and normalization
4. Ranking: TF-IDF, BERT, Logistic Regression, Random Forest
5. Explainability: SHAP (global) + LIME (local)
6. Evaluation metrics: Precision@K, Recall@K, MRR, nDCG
7. Ablation study
8. Fairness analysis

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

import config
print('Project root:', ROOT)
print('Active model:', config.ACTIVE_MODEL)
print('Section weights:', config.SECTION_WEIGHTS)
print('Score combination weights:', config.SCORE_COMBINATION_WEIGHTS)

## 2. Load Sample Data

In [ ]:
from data.sample.data_generator import save_ground_truth, GROUND_TRUTH, RELEVANCE_SCORES

gt = save_ground_truth()

job_path = config.SAMPLE_DIR / 'job_description.txt'
job_text = job_path.read_text(encoding='utf-8')
print(f'Job description: {len(job_text)} chars')

resume_files = sorted([
    f for f in config.SAMPLE_DIR.iterdir()
    if f.suffix == '.txt' and f.name.startswith('resume_')
])
print(f'Resume files ({len(resume_files)}):')
for f in resume_files:
    print(f'  {f.name}')

## 3. Text Preprocessing

In [ ]:
from preprocessing.pdf_parser import parse_resume_file
from preprocessing.text_cleaner import preprocess_for_tfidf, preprocess_for_bert
from preprocessing.section_extractor import extract_sections

raw = parse_resume_file(resume_files[0])
print('=== Raw text (first 400 chars) ===')
print(raw[:400])

print('\n=== TF-IDF preprocessed (first 250 chars) ===')
print(preprocess_for_tfidf(raw)[:250])

print('\n=== BERT preprocessed (first 250 chars) ===')
print(preprocess_for_bert(raw)[:250])

In [ ]:
sections = extract_sections(raw)
print('Extracted sections:')
for sec, content in sections.items():
    if sec != 'full_text' and content.strip():
        preview = content.strip()[:120].replace('\n', ' ')
        print(f'  [{sec.upper():15s}] {preview}...')

## 4. Skill Extraction & Normalization

In [ ]:
from skill_extraction.skill_extractor import get_skill_extractor

extractor = get_skill_extractor()

job_skills = extractor.extract_skills(job_text)
print(f'Job required skills ({len(job_skills)}):\n  {job_skills}\n')

print(f'{"Resume":<40} {"Skills":>6} {"Matched":>8} {"Jaccard":>8} {"F1":>7}')
print('-' * 74)
for rf in resume_files:
    text   = parse_resume_file(rf)
    skills = extractor.extract_skills(text)
    ov     = extractor.skill_overlap(skills, job_skills)
    print(f'{rf.name:<40} {len(skills):>6} {ov["overlap_count"]:>8} {ov["jaccard"]:>8.3f} {ov["f1"]:>7.3f}')

In [ ]:
# Alias normalization demo
aliases = ['py', 'sklearn', 'k8s', 'nodejs', 'tf', 'pyspark', 'gcp']
print('Skill alias normalizations:')
for a in aliases:
    print(f'  "{a}"  ->  "{extractor.ontology.normalize(a)}"')

## 5. TF-IDF Baseline Ranking

In [ ]:
from scoring.scorer import ResumeScorer, parse_job_description, parse_resume

job     = parse_job_description(job_text)
resumes = [parse_resume(rf) for rf in resume_files]

tfidf_scorer = ResumeScorer(model_name='tfidf')
tfidf_scored = tfidf_scorer.score_resumes(job, list(resumes))
tfidf_df     = tfidf_scorer.to_dataframe(tfidf_scored)

print('=== TF-IDF Rankings ===')
display(tfidf_df[['rank','file_name','final_score','tfidf_score','skill_overlap_score','matched_skills']].set_index('rank'))

## 6. BERT Semantic Ranking

In [ ]:
bert_scorer = ResumeScorer(model_name='bert')
bert_scored = bert_scorer.score_resumes(job, [parse_resume(rf) for rf in resume_files])
bert_df     = bert_scorer.to_dataframe(bert_scored)

print('=== BERT Rankings ===')
display(bert_df[['rank','file_name','final_score','bert_score','skill_overlap_score','matched_skills']].set_index('rank'))

## 7. Train ML Models (Logistic Regression + Random Forest)

In [ ]:
from data.sample.data_generator import generate_training_features
from models.ml_models import LogisticRegressionModel, RandomForestModel

X_train, y_train = generate_training_features()
print(f'Training set: {X_train.shape}, positives={int(y_train.sum())}, negatives={int((1-y_train).sum())}')

lr_model = LogisticRegressionModel()
lr_model.fit(X_train, y_train)
lr_model.save()
print('\nLogistic Regression - Top feature importances:')
for feat, imp in lr_model.get_feature_importance()[:7]:
    print(f'  {feat:<30s}: {imp:+.4f}')

rf_model = RandomForestModel()
rf_model.fit(X_train, y_train)
rf_model.save()
print('\nRandom Forest - Top feature importances:')
for feat, imp in rf_model.get_feature_importance()[:7]:
    print(f'  {feat:<30s}: {imp:.4f}')

In [ ]:
# Visualise feature importances side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (model, title) in zip(axes, [
    (lr_model, 'Logistic Regression\n(coefficient magnitude)'),
    (rf_model, 'Random Forest\n(Gini importance)'),
]):
    imps = model.get_feature_importance()[:10]
    names = [i[0] for i in imps]
    vals  = [abs(i[1]) for i in imps]
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(names)))
    ax.barh(names, vals, color=colors)
    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('ML Model Feature Importances', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. LR and RF Rankings

In [ ]:
for model_name in ['logistic_regression', 'random_forest']:
    scorer = ResumeScorer(model_name=model_name)
    scored = scorer.score_resumes(job, [parse_resume(rf) for rf in resume_files])
    df     = scorer.to_dataframe(scored)
    print(f'\n=== {model_name.upper()} Rankings ===')
    print(df[['rank','file_name','final_score','bert_score','skill_overlap_score']].to_string(index=False))

## 9. Score Comparison Across All Models

In [ ]:
results = {}
for model_name in ['tfidf', 'bert', 'logistic_regression', 'random_forest']:
    scorer = ResumeScorer(model_name=model_name)
    scored = scorer.score_resumes(job, [parse_resume(rf) for rf in resume_files])
    results[model_name] = {r.file_name: r.final_score for r in scored}

compare_df = pd.DataFrame(results)
compare_df.index = [idx.replace('resume_','').replace('.txt','') for idx in compare_df.index]
compare_df = compare_df.sort_values('bert', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(compare_df))
w = 0.2
colors = ['#3498db','#e74c3c','#27ae60','#f39c12']
for i, (col, c) in enumerate(zip(compare_df.columns, colors)):
    ax.bar(x + i*w, compare_df[col], w, label=col, color=c, alpha=0.85)
ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(compare_df.index, rotation=20, ha='right')
ax.set_ylabel('Final Score')
ax.set_title('Score Comparison Across All Models', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

display(compare_df.round(4))

## 10. Explainability — SHAP Global Feature Importance

In [ ]:
from explainability.shap_explainer import SHAPExplainer
from models.ml_models import FEATURE_NAMES

# Use the RF model for SHAP (TreeExplainer is most accurate)
scorer  = ResumeScorer(model_name='random_forest')
scored  = scorer.score_resumes(job, [parse_resume(rf) for rf in resume_files])

fvs = [r.feature_vector for r in scored if r.feature_vector is not None]
if fvs:
    X_scored = np.stack(fvs)
    shap_exp = SHAPExplainer(model=scorer._ml_model, model_type='random_forest')
    shap_exp.fit(X_scored, feature_names=FEATURE_NAMES)
    shap_vals = shap_exp.explain(X_scored)
    plot_path = shap_exp.plot_summary(
        X_scored,
        save_path=config.PLOTS_DIR / 'shap_global_nb.png',
        title='SHAP Global Feature Importance (Random Forest)'
    )
    from IPython.display import Image
    display(Image(filename=str(plot_path)))
    print('\nGlobal feature importance (top 10):')
    for feat, val in shap_exp.get_global_importance()[:10]:
        print(f'  {feat:<30s}: {val:.5f}')
else:
    print('No feature vectors available (RF model may not be fitted).')

## 11. Explainability — LIME Local Explanation

In [ ]:
from explainability.lime_explainer import LIMEExplainer

if fvs and scorer._ml_model and scorer._ml_model.is_fitted:
    lime_exp = LIMEExplainer(feature_names=FEATURE_NAMES)

    # Explain the top-ranked resume
    top_resume = scored[0]
    if top_resume.feature_vector is not None:
        explanation = lime_exp.explain_instance(
            instance=top_resume.feature_vector,
            predict_fn=lambda X: scorer._ml_model.predict_proba(X),
            training_data=X_scored,
            resume_id=top_resume.resume_id,
        )
        if explanation:
            print(lime_exp.format_explanation_text(explanation))
            lime_path = lime_exp.plot_explanation(
                explanation,
                save_path=config.PLOTS_DIR / f'lime_{top_resume.resume_id}_nb.png'
            )
            if lime_path:
                display(Image(filename=str(lime_path)))
else:
    print('Skipped: RF model not fitted or no feature vectors.')

## 12. Full Explainability Pipeline (Skill Analysis Plots)

In [ ]:
from explainability.explainer_pipeline import ExplainabilityPipeline
from IPython.display import Image

bert_scorer = ResumeScorer(model_name='bert')
bert_scored = bert_scorer.score_resumes(job, [parse_resume(rf) for rf in resume_files])

xai = ExplainabilityPipeline(model_type='bert')
explanations = xai.explain_all(bert_scored, job, output_dir=config.PLOTS_DIR)

# Display skill analysis for top 3 candidates
for resume in bert_scored[:3]:
    exp = explanations.get(resume.resume_id, {})
    print(f'\n{"="*60}')
    print(exp.get('narrative', 'No narrative.'))
    skill_plot = exp.get('skill_plot')
    if skill_plot and Path(skill_plot).exists():
        display(Image(filename=skill_plot))

## 13. Evaluation Metrics

In [ ]:
from evaluation.metrics import compute_ranking_metrics, format_metrics_report

relevant_ids = GROUND_TRUTH['job_description.txt']['relevant_resumes']
print(f'Ground truth relevant resumes: {relevant_ids}\n')

all_metrics = {}
for model_name in ['tfidf', 'bert', 'logistic_regression', 'random_forest']:
    scorer = ResumeScorer(model_name=model_name)
    scored = scorer.score_resumes(job, [parse_resume(rf) for rf in resume_files])
    ranked_ids = [r.file_name for r in scored]
    metrics = compute_ranking_metrics(ranked_ids, relevant_ids, config.EVAL_K_VALUES)
    all_metrics[model_name] = metrics
    print(f'--- {model_name.upper()} ---')
    print(format_metrics_report(metrics))

In [ ]:
# Metrics comparison bar chart
metric_keys = ['precision@3', 'recall@3', 'ndcg@3', 'mrr', 'map']
models = list(all_metrics.keys())

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(metric_keys))
w = 0.18
colors = ['#3498db','#e74c3c','#27ae60','#f39c12']
for i, (m, c) in enumerate(zip(models, colors)):
    vals = [all_metrics[m].get(k, 0) for k in metric_keys]
    ax.bar(x + i*w, vals, w, label=m, color=c, alpha=0.85)

ax.set_xticks(x + 1.5*w)
ax.set_xticklabels([k.upper() for k in metric_keys])
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_title('Evaluation Metrics Across Models', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'eval_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Ablation Study

In [ ]:
from evaluation.metrics import ablation_study

ablation_df = ablation_study(
    job_text=job_text,
    resume_files=resume_files,
    relevant_ids=relevant_ids,
    output_path=config.REPORTS_DIR / 'ablation_study.csv',
)

print('=== Ablation Study Results ===')
display(ablation_df.round(4))

In [ ]:
# Ablation heatmap
import seaborn as sns

plot_cols = [c for c in ablation_df.columns if c != 'model']
fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(
    ablation_df[plot_cols].astype(float),
    annot=True, fmt='.3f', cmap='YlGnBu',
    linewidths=0.5, ax=ax, vmin=0, vmax=1,
)
ax.set_title('Ablation Study — Metrics Heatmap', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'ablation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Fairness Analysis

In [ ]:
from evaluation.fairness import run_fairness_analysis, format_fairness_report

bert_scorer = ResumeScorer(model_name='bert')
bert_scored = bert_scorer.score_resumes(job, [parse_resume(rf) for rf in resume_files])
bert_df     = bert_scorer.to_dataframe(bert_scored)

fairness = run_fairness_analysis(
    bert_df,
    output_path=config.FAIRNESS_LOG_PATH,
    plot_path=config.PLOTS_DIR / 'fairness_nb.png',
)

print(format_fairness_report(fairness))
display(Image(filename=str(config.PLOTS_DIR / 'fairness_nb.png')))

## 16. Save All Results

In [ ]:
import json

# Save metrics summary
summary_path = config.REPORTS_DIR / 'metrics_summary.json'
with open(summary_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f'Metrics summary saved: {summary_path}')

# Save rankings CSV
rankings_path = config.RANKINGS_DIR / 'bert_rankings.csv'
bert_df.to_csv(rankings_path, index=False)
print(f'Rankings CSV saved: {rankings_path}')

print('\nOutput files:')
for p in sorted(config.OUTPUTS_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(config.ROOT_DIR)}')